## Exploration Notebook

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from src.context_builder import build_context
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### Testing Groq API

In [5]:
#adding Groq API
load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"))

In [5]:
#testing that LLM works with easy query
response = llm.invoke("What is a good gaming headset?")
print(response.content)

There are many great gaming headsets available, depending on your budget, gaming platform (e.g., PC, Xbox, PlayStation, or Nintendo Switch), and personal preferences. Here are some popular options:

**Budget-friendly (Under $100)**

1. HyperX Cloud II: A popular, affordable headset with excellent sound quality and a comfortable design.
2. Turtle Beach Recon 200: A budget-friendly option with a strong focus on comfort and a sleek design.
3. SteelSeries Arctis 3: A lightweight, foldable headset with good sound quality and a stylish design.

**Mid-range ($100-$250)**

1. SteelSeries Arctis 7: A high-end version of the Arctis 3, with wireless connectivity and a more premium design.
2. HyperX Cloud Flight: A wireless headset with long battery life and excellent sound quality.
3. Astro Gaming A40: A high-end headset with excellent sound quality and a comfortable design.

**High-end ($250-$500)**

1. Sennheiser GSP 670: A premium headset with excellent sound quality, a comfortable design, and

### Prompt exploration: Semantic RAG

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local('../data/processed/faiss_index', embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
#prompt testing

#trying with a less detailed prompt, minimal instructions
prompt_1 = ChatPromptTemplate.from_template("""
You are a helpful Amazon video game shopping assistant.
Answer using ONLY the context. Be concise and relevant with your product recommendations.

Context: {context}
Question: {input}
Answer:
""")

#more detailed prompt of exactly what to make product recommendations on
prompt_2 = ChatPromptTemplate.from_template("""
You are a helpful Amazon video game product shopping assistant.
Answer the question using ONLY the information in the context given.
The context contains real Amazon product reviews and metadata (product title, rating, price, description etc.).

- Recommend specific products that match the user's query
- Support your recommendation with concise and relevant evidence from the reviews and ratings
- Consider important product aspects like price and rating
- Only say you don't know if the context truly does not contain relevant information

Context: {context}
Question: {input}
Answer:
""")

In [16]:
#testing pipeline with different prompts on medium-level query 

query = "What is a beginner friendly game for ps4?"

for i, p in enumerate([prompt_1, prompt_2], 1):
    chain = (
        {"context": retriever | build_context, "input": RunnablePassthrough()}
        | p
        | llm
        | StrOutputParser()
    )
    print(f"\n{'='*60}\nPrompt {i}\n{'='*60}") #adding divider to output for easier prompt comparison
    print(chain.invoke(query))


Prompt 1
Based on the reviews, I would recommend "Forager" for a beginner-friendly game on PS4. It has a high rating of 4.6/5 and is described as a fun, simple, and relaxing game that's great for the entire family. The review mentions that it's "addictive" and that the player's grandson loves it, so it's likely to be engaging and easy to pick up for beginners.

Prompt 2
Based on the provided reviews and metadata, I recommend "Forager" for PS4 as a beginner-friendly game. 

Evidence from the review supports this recommendation: 

- The reviewer states that "this game is simply amazing it is quite fun for the entire family" which implies that it's suitable for players of different skill levels, including beginners.
- They also mention that it's a "great game for the PS4 family friendly and fun" which further supports its beginner-friendliness.
- Additionally, the game's description mentions that it "starts small and improves your base skills equipment network of friends and enemies" whi

- Prompt 1: more concise output with 1 recommendation. Links user query with relevant reviews and keywords from reviews.
- Prompt 2: more detailed output than prompt 1. More reasoning/justification of product recommendation. Provides a summary of choice and compares to other products, which could be useful to the user.

In [9]:
# prompt 3 - having LLM recommend multiple products

prompt_3 = ChatPromptTemplate.from_template("""
You are a helpful Amazon video game product shopping assistant.
Answer the question using ONLY the information in the context given.
The context contains real Amazon product reviews and metadata (product title, rating, price, description etc.).

- Recommend up to 3 products that match the user's query
- Support your recommendation with evidence from the reviews and ratings
- Consider important product aspects like price and rating
- Only say you don't know if the context truly does not contain relevant information

Context: {context}
Question: {input}
Answer:
""")

In [11]:
query2 = "What are some fun video games for sports lovers?"

chain2 = (
    {
        "context": retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt_3
    | llm
    | StrOutputParser()
    )

print(f"\n{'='*60}\nPrompt 3\n{'='*60}")
print(chain2.invoke(query2))


Prompt 3
Based on the context you've provided, I'd be happy to recommend some fun video games for sports lovers. Here are three options that match your query:

1. **Instant Sports Plus (Nintendo Switch)**: This game has a rating of 4.3/5 and is described as the most complete experience of the Instant Sports series, with 15 different sports and activities to enjoy. The reviewer "basic game play" found it to be a "cute" game, and another reviewer mentioned it's "fun for my 10yr old." Since it supports up to 4 local players, it's perfect for multiplayer fun. (Rating: 4.3/5, Price: Unknown)

2. **Sports Party (Nintendo Switch)**: Although this game received a mixed review from one user, who was looking for a motion movement game, it still has a rating of 4.2/5. The reviewer mentioned it's "fun" and their 8-year-old grandson loves it. It's a great option if you're looking for a game that's accessible to anyone. (Rating: 4.2/5, Price: $24.08)

3. **Instant Sports Allstars for Nintendo Switc

In [12]:
#what if the user asks for 1 product, will LLM recognize this or will it recommend 3 regardless?
query3 = 'Xbox controller with the best battery life'

print(chain2.invoke(query3))

Based on the context provided, I recommend the following Xbox controller products with the best battery life:

1. **Venom Xbox Controller Rechargeable Battery Twin Pack**:
This product has a 4.4/5 rating and is priced at $26.24. According to the reviews, it has "decent battery life" and can last up to 15 hours from a full charge. Although one reviewer mentioned that one of the batteries died quickly, the overall reviews suggest that this product has good battery life.

2. **Xbox Controller with 1400mAh Lithium Battery**:
This product has a 4.4/5 rating and is priced at $39.98. Although one reviewer mentioned that the buttons are misplaced, there are no comments about the battery life. However, the product description mentions a 1400mAh lithium battery, which suggests that it has a decent battery life.

3. **Theocleo Black 3600mAh Replacement Battery for Xbox 360 Controller**:
This product has a 5.0/5 rating and is priced at an unknown price (since the price is not mentioned in the cont

- Prompt 3 also performs well, recommending relevant products that match user's query. 
- Seems to always recommend top 3 - might be redundant since UI/web app will already have top 3 products displayed.

**Selected prompt**: Prompt 2 - See rational in [milestone2_discussion.md](https://github.com/UBC-MDS/DSCI_575_project_nourabd_mailysg/blob/main/results/milestone2_discussion.md)

### Testing Implemented Semantic RAG

In [18]:
from src.semantic_rag_pipeline import get_semantic_rag_chain

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
semantic_chain = get_semantic_rag_chain()

queries = [
    "What are some good accessories to help with my gaming set up", #more complex 
    "Multi player game for nintendo switch", #medium 
    "Good ps3 controller" #easier
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    print(semantic_chain.invoke(query))


Query: What are some good accessories to help with my gaming set up
Based on the provided context, I recommend the following accessories to help with your gaming setup:

1. **Brainwavz EOL The Dock** (Rating: 4.4/5): This under-desk mount for Xbox Series X, PS5, PS4, Switch, and PC gamepads is a great option. While it has some mixed reviews, many customers have found it to be sturdy and effective. One reviewer mentioned that it's "exactly what I wanted" and another said it's a "solid under-desk plastic" that "keeps my PS4 controller in place." (Price: Not available)

2. **Game Controller Holder Wall Mount 2 Pack** (Rating: 4.6/5): This universal design is suitable for most modern and retro game controllers, including Xbox, PS4, PS5, Switch, and PC gamepads. Reviewers have found it to be "sturdy" and "serve(s) its purpose well enough." (Price: Not available)

3. **Cooler Master Dual-Sided Gaming Mouse Pad** (Rating: 4.3/5, Price: $17.96): This gaming mouse pad is a great accessory for 

Semantic rag pipeline working as expected using a mix of different queries. Quality of products recommended is good and matches user's request, with justification, mention of pricing and overall rating, with good, concise links to relevant reviews.

### Qualitative Evaluation for Hybrid RAG Workflow

Running the Hybrid RAG pipeline for the queries created in Milestone 1:

In [31]:
from src.hybrid_rag_pipeline import hybrid_rag_pipeline, get_retriever

In [38]:
# Milestone 1 queries
queries = [
        # Easy
        "nintendo switch screen protector",
        "gaming headset with microphone",
        "xbox 360 wireless controller",
        "fifa 16 game for playstation",
        # Medium
        "playstation games for sports lovers",
        "competitive multiplayer game for large groups",
        "comfortable headset for long gaming sessions",
        "ps4 controller with good battery life",
        # Complex
        "best wireless gaming headset under $100",
        "good single player story game for a beginner who just got their first playstation under $50"
    ]

In [33]:
retriever = get_retriever('../')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from ../data/processed/faiss_index
Loading retriever...
retriever loaded from ../data/processed/bm25_index
Creating hybrid retriever...
Completed hybrid retriever!


#### Easy (Keyword-based) Queries :

In [20]:
query = queries[0] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: nintendo switch screen protector
Based on the given context, I would recommend the following products:

1. **evoretro Screen Protector Compatible for Nintendo Switch Lite**: This product has a rating of 4.6/5 and is priced at $12.5. It has received great reviews for its ease of installation, durability, and high accuracy response. Many reviewers have mentioned that it fits perfectly on the Switch Lite and provides great protection against damage.

   Review Excerpts:
   - "Easy application and it fits perfectly."
   - "It looks great on the switch, you cant even tell theres a screen protector on it."
   - "Works great, easy to apply and lasts forever."

2. **Yonntech 3 Pack Screen Protector for Nintendo Switch Lite**: This product has a rating of 4.6/5 but the price is not mentioned. It has received great reviews for its ease of application and ability to protect the screen. Some reviewers have mentioned that it doesn't completely cover the screen, but it still does its job wel

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?| yes |
|Completeness|Does the answer address all aspects of the question?| yes |
|Fluency|Is the answer natural, clear, and easy to read?| yes|

In [21]:
query = queries[1] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: gaming headset with microphone
Based on your query, I have found a few products that match your requirements. Since the prices are not specified, I will focus on the ratings and reviews.

1. **Ideazon GH100 Gaming Headset**: This product has a rating of 4.4/5 and has received positive reviews for its sound quality. Although the mic is uncomfortable and the cord broke easily, it's still a good option to consider.

2. **Lesozoh A10 Gaming Headset**: This product has a rating of 3.8/5 and is priced at $27.97. Reviews praise its excellent performance, high-definition speaker, and vibration unit. However, the mic is not working for some users.

3. **Gaming Headset Headphones with Microphone**: This product has a rating of 3.9/5 but the reviews are mixed. Some users praise its comfort, good mic, and RGB light, while others mention that the wires break easily and the product came broken.

Considering the ratings and reviews, I would recommend the **Ideazon GH100 Gaming Headset** as th

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?| yes |
|Completeness|Does the answer address all aspects of the question?|no|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

In [22]:
query = queries[2] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: xbox 360 wireless controller
Based on your query, I recommend the following products:

1. **Product: wireless controller for xbox 360 wireless controller remote 24ghz game controller gamepad joystick for xboxslim 360 pc windows 7810 black** 
Rating: 4.4/5
Price: Not Available
Description: This product has received positive reviews, with users praising its quality, ease of use, and value for money. Many reviewers have mentioned that it works great and is a good replacement for the original Xbox 360 controller.

2. **Product: yccteam wireless controller for xbox 360 wireless game controller gamepad joystick with vibration shoulder buttons for microsoft xbox 360xbox 360 slimblack** 
Rating: 4.4/5
Price: 17.33
Description: This product is another great option for an additional wireless controller for your Xbox 360. Reviewers have mentioned that it's convenient, easy to use, and has good quality. It's also a great value for the price.

Both of these products seem to be great options

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|yes|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

In [23]:
query = queries[3] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: fifa 16 game for playstation
Based on the provided context, I recommend the "fifa 16  deluxe edition  playstation 4" as the best option for the FIFA 16 game on PlayStation.

This product has a rating of 4.3/5 and a price of $13.95. The reviews suggest that the game is enjoyable, with many reviewers stating that their children and grandchildren love playing it. Some reviewers mention that the game is fun, interactive, and has great graphics. However, there are some negative reviews mentioning that the redemption code is invalid, and the game may not be as good as previous versions.

Overall, the positive reviews and ratings suggest that this product is a good option for those looking for a fun and interactive FIFA game on PlayStation.


|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|yes|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

#### Medium (Semantic-based) Queries :

In [24]:
query = queries[4] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: playstation games for sports lovers
Based on your query, I'd recommend the following PlayStation games for sports lovers:

1. **Sports Champions for PlayStation 3**: This game has an average rating of 4.6/5 and a price of $29.99. It's a fun and physically engaging game that lets you and your friends compete in various sports events using the PlayStation Move controller. Reviewers have praised its fun and addictive gameplay, great graphics, and excellent use of the Move controller.
2. **Sports Champions 2 for PlayStation 3**: This game has an average rating of 4.6/5 and a price of $39.94. It's a sequel to the original Sports Champions and offers even more fun and challenging sports games, including bowling, tennis, and archery. Reviewers have praised its improved graphics, new games, and great family entertainment value.
3. **Sports Champions for PlayStation 3 (Refurbished)**: This game has an average rating of 4.8/5 and a price of $11.26. It's a great value option that offers t

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|yes|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

In [25]:
query = queries[5] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: competitive multiplayer game for large groups
Based on the context provided, I would recommend the "Ace of Spades Battle Builder" as a competitive multiplayer game for large groups.

Here's why:

- The game allows up to 32 players in a single match, which is a large group.
- The game description mentions that it's a "first-person shooter that lets you create your battleground, destroy it, then create it again" which implies competitive gameplay.
- Although one of the reviews mentions that the game crashes, another review states that it's an "awesome game" that's like "Minecraft meets Counter-Strike", which suggests that it's a game worth considering.

While there are some concerns about the game's stability, it seems to offer the competitive multiplayer experience that you're looking for.


|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|yes|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

In [26]:
query = queries[6] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: comfortable headset for long gaming sessions
Based on your query, I would recommend the following products that match your requirement for a comfortable headset for long gaming sessions:

1. **Tatybo Wireless Gaming Headset for PS4 PS5 PC**: This product has a 4.6/5 rating and is priced at $49.99. Reviewers have praised it for being "super comfortable", "easy to use", and having "good quality" sound. One reviewer even mentioned that it's "comfortable for long gaming sessions".

2. **Fantech Shipping Cost**: Although the price is not mentioned, this product has a 5.0/5 rating. Reviewers have praised it for being "very comfortable" and having "great sound quality". This suggests that it's a good option for long gaming sessions.

3. **Rehoc 35mm Gaming Headset Headphones Surround Sound Stereo Wired Earphone for PC Notebook**: This product has a 5.0/5 rating. Reviewers have praised it for being "very comfortable" and "rarely uncomfortable". Although there's no specific mention of l

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|yes|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

In [27]:
query = queries[7] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: ps4 controller with good battery life
Based on the reviews and ratings provided, I highly recommend the "Pickle Power 2 Pack 2000mah PS4 PS4 Pro Battery Replacement" product (Rating: 4.4/5, Price: $17.89). This product is specifically designed to replace the batteries in your PS4 controller, and the reviews suggest that it greatly improves the battery life of the controller.

Multiple reviewers have stated that the batteries work perfectly, last most of the day, and are a great upgrade to extend the controller's life. Some reviewers have even mentioned that the batteries work as well as the original ones and have increased their playtime significantly.

While the other products reviewed do have good battery life, they are not specifically designed as battery replacements, and the reviews do not provide the same level of detail and satisfaction as the Pickle Power product.

The other products that may be worth considering are:

* Vidppluing Wireless Controller for PS4 (Rating: 4

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|no|
|Completeness|Does the answer address all aspects of the question?|no|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

#### Complex Queries :

In [39]:
query = queries[8] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: best wireless gaming headset under $100
Based on the reviews and ratings provided, I recommend the following wireless gaming headsets under $100:

1. **Plantronics Rig 800LX Wireless Gaming Headset for Xbox One**:
	* Rating: 4.2/5
	* Price: $69.99
	* Reviewer praise: "awesome sound", "great deal", "comfortable", "long battery life"
	* Consider: This headset has received positive reviews for its sound quality, comfort, and battery life. However, some reviewers have reported issues with the microphone and connectivity.
2. **Corsair Void RGB Elite Wireless Premium Gaming Headset with 7.1 Surround Sound (Carbon)**:
	* Rating: 4.4/5
	* Price: $73.95 (renewed)
	* Reviewer praise: "amazing sound", "great quality", "fantastic headset", "comfortable"
	* Consider: This headset has received high praise for its sound quality, comfort, and build quality. However, some reviewers have reported issues with the microphone and battery life.

While both headsets are under $100, the Plantronics Ri

|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|no|
|Fluency|Is the answer natural, clear, and easy to read?|yes|

In [36]:
query = queries[9] 
print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)
print(hybrid_rag_pipeline(retriever, query))


Query: good single player story game for a beginner who just got their first playstation under $50
Based on the context provided, I recommend the product "Vikings: Wolves of Midgard" for PlayStation 4.

Rating: 4.2/5
Price: $24.97

Reviewers have praised the game for its:
- Fun single-player experience (e.g., "single player is fun", "enjoyable game exactly as described", "its extremely fun to play and very appealing to female gamers")
- Great graphics and a few puzzles (e.g., "fantastic game preorder top of the line action rpg awesome timing service and perfect condition")
- Decent game for a beginner (e.g., "even a gaming beginner can have fun with this game")

Additionally, the reviewer who mentioned the game as repetitive and short, also mentioned that it's "not a bad game for $25" which aligns with your price criteria.

Please note that the game is for PlayStation 4 and not the original PlayStation.


|Dimension | What to assess | yes/no |
|----------|----------------|--------|
|Accuracy|Is the answer factually correct based on the reviews?|yes|
|Completeness|Does the answer address all aspects of the question?|yes|
|Fluency|Is the answer natural, clear, and easy to read?|yes|